In [1]:
from bs4 import BeautifulSoup
from bs4 import Tag
import json
import os
import tqdm
from selenium.webdriver.chrome.service import Service
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
import requests

In [2]:
l=[
    ("https://www.aparat.com/v/jsPIg?playlist=1041759","dars_hafez.html",'درس‌گفتار حافظ‌شناسی'),
    ("https://www.aparat.com/v/tRhOI?playlist=6411553","fekr_saeb.html",'فکر صائب'),
    ("https://www.aparat.com/v/yphvH?playlist=881126","jam_hafez.html",'جامِ جهان‌نمایِ حافظ'),
]

In [3]:
def parse_item(item:Tag):
    prefix="https://www.aparat.com"
    poster=item.select_one('.poster img')['src']
    vid_url=item.select_one('.thumb-content a')['href']
    title=item.select_one('.title span').text
    return poster,prefix+vid_url,title

In [4]:
links=[]

In [5]:
def get_download_link(link:str):
    # print(link)
    video_id=link.split('\\')[-1].split('/')[-1].split("?")[0]
    print(video_id)
    api_url = f"https://www.aparat.com/etc/api/video/videohash/{video_id}/get"

    chrome_driver_path = "C:\\chromedriver.exe"
    s=Service(executable_path=chrome_driver_path)
    # Launch Chrome with the driver
    driver = webdriver.Chrome(service=s)

    # Navigate to a website
    driver.get(api_url)
    # print(driver.page_source)
    content=driver.find_element(By.TAG_NAME,'pre').text
    # response = requests.get(api_url)
    # data = response.json()
    data = json.loads(content)

    with open("tmp.json",'w',encoding='utf-8')as f:
        f.write(driver.page_source)
    if data['video']:
        # videos = data["video"]["file_link_all"]
        return data["video"]["file_link"]
        # for i in videos:
        #     if i['profile']=='720p':return i['urls'][0]
        # else:
        #     return videos[-1]['urls'][0]
    return ""

In [6]:
def html_to_parsed_json(file_name,output_file,category):
    l=[]
    os.makedirs("jsons",exist_ok=True)
    with open(file_name,encoding="utf8") as fp:
        soup = BeautifulSoup(fp, "html.parser")
    items=soup.select('.item-playlist')
    for i in items:
        poster,vid_url,title=parse_item(i)
        # vid_down_url=get_download_link(vid_url)
        vid_down_url=""
        d={"poster":poster,"vid_url":vid_url,"title":title,"info":"","vid_down_url":vid_down_url}
        f_n=file_name.split('/')[-1].split("\\")[-1].split('.')[0].replace('_','-')
        l.append({"data":d,"category":category,"category_fing":f_n})
    of=output_file if output_file.endswith('.json') else output_file+".json"
    with open(os.path.join('jsons',of), "w",encoding='utf8') as f:
        json.dump(l,f,ensure_ascii=False)

In [7]:
for i in l:
    i=i[1:]
    f_n=os.path.join('htmls',i[0])
    j_name=i[0].replace('.html','.json')
    category=i[1]
    html_to_parsed_json(f_n,j_name,category)

In [8]:
conc_l=[]
for i in os.listdir("jsons/"):
    with open(os.path.join("jsons",i) ,encoding='utf-8') as f:
        obj=json.load(f)
        conc_l+=(obj)
with open("result.json", "w",encoding='utf8') as f:
    json.dump(conc_l,f,ensure_ascii=False)        

In [9]:
conc_l.__len__()

44

In [10]:
!cp result.json ../src/assests/result.json